In [1]:
!pip install transformers
!pip install sentence-transformers
!pip install nltk
!pip install spacy
!pip install scikit-learn
!pip install pandas
!pip install pyngrok

In [2]:
import pandas as pd
import numpy as np
import re

import nltk
import spacy

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

from transformers import pipeline

In [3]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [4]:
resume_data = {

    "role":[

        "Data Analyst",
        "Data Scientist",
        "NLP Engineer",
        "Business Analyst",
        "ML Engineer"

    ],

    "skills":[

        "Excel SQL Tableau Power BI",

        "Python ML Statistics NLP",

        "Python NLP Transformers LLM",

        "Excel SQL Power BI Analytics",

        "Python ML Deep Learning"

    ]

}

resume_df = pd.DataFrame(resume_data)

resume_df

,role,skills
0,Data Analyst,Excel SQL Tableau Power BI
1,Data Scientist,Python ML Statistics NLP
2,NLP Engineer,Python NLP Transformers LLM
3,Business Analyst,Excel SQL Power BI Analytics
4,ML Engineer,Python ML Deep Learning


In [5]:
resume_df.shape

(5, 2)

In [6]:
resume_df.isnull().sum()

,0
role,0
skills,0


In [7]:
resume_df['role'].value_counts()

,count
role,
Data Analyst,1
Data Scientist,1
NLP Engineer,1
Business Analyst,1
ML Engineer,1


In [8]:
resume_df["clean_skills"] = (

    resume_df["skills"].str.lower()
)

In [9]:
def clean_text(text):

    text = re.sub(

        r'[^a-zA-Z\s]',

        '',

        text

    )

    return text

In [10]:
resume_df["clean_skills"] = (

    resume_df["clean_skills"]

    .apply(clean_text)

)

In [11]:
resume_df["tokens"] = (

    resume_df["clean_skills"]

    .apply(lambda x: x.split())

)

In [12]:
stop_words = set(

    stopwords.words("english")

)

def remove_stopwords(tokens):

    return [

        word

        for word in tokens

        if word not in stop_words

    ]

resume_df["filtered_tokens"] = (

    resume_df["tokens"]

    .apply(remove_stopwords)

)

In [13]:
stemmer = PorterStemmer()

resume_df["stemmed"] = (

    resume_df["filtered_tokens"]

    .apply(

        lambda x:[

            stemmer.stem(i)

            for i in x

        ]

    )

)

In [14]:
lemmatizer = WordNetLemmatizer()

resume_df["lemmatized"] = (

    resume_df["filtered_tokens"]

    .apply(

        lambda x:[

            lemmatizer.lemmatize(i)

            for i in x

        ]

    )

)

In [16]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [17]:
sample = resume_df["tokens"].iloc[0]

nltk.pos_tag(sample)

[('excel', 'NN'),
 ('sql', 'NN'),
 ('tableau', 'NN'),
 ('power', 'NN'),
 ('bi', 'NN')]

In [18]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 89.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
nlp = spacy.load(

    "en_core_web_sm"

)

text = """

Pranay Shukla worked at Infosys

"""

doc = nlp(text)

for ent in doc.ents:

    print(

        ent.text,

        ent.label_

    )

Pranay Shukla PERSON
Infosys ORG


In [20]:
tfidf = TfidfVectorizer()

X = tfidf.fit_transform(

    resume_df["clean_skills"]

)

keywords = (

    tfidf

    .get_feature_names_out()

)

print(keywords)

['analytics' 'bi' 'deep' 'excel' 'learning' 'llm' 'ml' 'nlp' 'power'
 'python' 'sql' 'statistics' 'tableau' 'transformers']


In [21]:
role_skill_map = {

"Data Scientist":[

"Python",
"Machine Learning",
"Deep Learning",
"NLP",
"Statistics",
"Tableau"

],

"Data Analyst":[

"Excel",
"SQL",
"Power BI",
"Tableau"

]

}

In [22]:
target_role = "Data Scientist"

role_skill_map[target_role]

['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'Statistics', 'Tableau']

In [23]:
embedding_model = SentenceTransformer(

'all-MiniLM-L6-v2'

)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
roles = resume_df["role"].tolist()

role_embeddings = (

    embedding_model

    .encode(roles)

)

In [26]:
user_role = "AI Engineer"

user_embedding = (

    embedding_model

    .encode([user_role])

)

similarities = cosine_similarity(

    user_embedding,

    role_embeddings

)

best_role = roles[

    similarities.argmax()

]

print(best_role)

ML Engineer


In [27]:
def ats_score(

    user_skills,

    target_role

):

    required = set(

        role_skill_map[target_role]

    )

    user = set(user_skills)

    score = (

        len(

            required.intersection(user)

        )

        /

        len(required)

    ) * 100

    return round(score,2)

In [28]:
generator = pipeline(

"text-generation",

model="gpt2"

)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [29]:
prompt = """

Create a professional resume summary.

Skills:

Python
SQL
Machine Learning

Role:

Data Scientist

"""

summary = generator(

prompt,

max_length=120

)

print(

summary[0]["generated_text"]

)

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=120) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Create a professional resume summary.

Skills:

Python
SQL
Machine Learning

Role:

Data Scientist


Background:

After graduating from Harvard Business School in 2009, I worked for a private firm. Since then I've been developing and providing data analysis services and services for the Fortune 500 companies, from SAP to IBM, to Fortune 500 enterprises.

How to apply:

Go to the "About Us" section at Google or Facebook.

Submit your resume to The Product Hunt.

Click "Apply Now" to submit your resume.

For more information, go to the "About Us" section at Google or Facebook.

In order to apply, please go to the "About Us" section at Google or Facebook.

For more information, go to the "About Us" section at Google or Facebook.

In order to apply, please go to the "About Us" section at Google or Facebook.

For more information, go to the "About Us" section at Google or Facebook.

For more information, go to the "About Us" section at Google or Facebook.

For more information, go to the 

In [30]:
prompt = """

Create a professional resume summary.

Skills:

Python
SQL
Machine Learning

Role:

Data Scientist

"""

summary = generator(

prompt,

max_length=120

)

print(

summary[0]["generated_text"]

)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=120) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Create a professional resume summary.

Skills:

Python
SQL
Machine Learning

Role:

Data Scientist


Background:

Computer science is a discipline where you will often teach yourself skills to help you make better decisions. I'm always interested in learning more about the human and the machine, and I'm fortunate enough to be a strong believer in the power of computer science.

For me, the biggest challenge, however, is understanding why people are different.


I'm a computer scientist, however, I'm not particularly good at this. I'm usually only good at one thing: the ability to see what things people do. What if I could see what people actually do? Well, that would be pretty cool.


For me, I've always been able to see things that have been done. In other words, I can see things I've done that I can't, or don't want to see, that I can't see, that I can't see. So, for example, I can see how much money I made on my last job, and then I can see what my total wealth is.


So, why would

In [31]:
def generate_resume(

name,

role,

skills,

education,

projects

):

    resume = f"""

NAME:
{name}

TARGET ROLE:
{role}

SKILLS:
{skills}

EDUCATION:
{education}

PROJECTS:
{projects}

"""

    return resume

In [32]:
generate_resume(

"Pranay Shukla",

"Data Scientist",

"Python, SQL, NLP",

"MBA Analytics",

"AI Chatbot Platform"

)

'\n\nNAME:\nPranay Shukla\n\nTARGET ROLE:\nData Scientist\n\nSKILLS:\nPython, SQL, NLP\n\nEDUCATION:\nMBA Analytics\n\nPROJECTS:\nAI Chatbot Platform\n\n'

In [33]:
%%writefile app.py

import streamlit as st

st.title(

"AI Resume Builder"

)

name = st.text_input(

"Name"

)

role = st.selectbox(

"Role",

[
"Data Analyst",
"Data Scientist",
"NLP Engineer"
]

)

skills = st.text_area(

"Skills"

)

education = st.text_area(

"Education"

)

projects = st.text_area(

"Projects"

)

if st.button(

"Generate Resume"

):

    st.write(

f"""

# {name}

## {role}

Skills:
{skills}

Education:
{education}

Projects:
{projects}

"""

)

Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

/bin/bash: line 1: streamlit: command not found
⠙⠹⠸⠼⠴⠦⠧⠇⠏Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴your url is: https://four-parts-make.loca.lt
y
y


In [ ]:
y